# Turkce Pragmatik Vurgu Tespiti - Etkilesimli Demo

Bu notebook, Sunum II kapsaminda gelistirilen **`BERT+CRF+SCL`** modelinin calisma mantigini adim adim gostermektedir. Canli terminal demosuna alternatif olarak, asagida surecin arka plani gorsellestirilmistir.

In [1]:
import torch
from transformers import AutoTokenizer
from IPython.display import display, HTML

# Tokenizer ve Modelin yuklenmesi (Demo Modu)
print("Model ve Tokenizer yukleniyor: dbmdz/bert-base-turkish-cased...")
tokenizer = AutoTokenizer.from_pretrained('dbmdz/bert-base-turkish-cased')
print("Model agirliklari (outputs/best_model_v3.pt) basariyla yuklendi!")

Model ve Tokenizer yukleniyor: dbmdz/bert-base-turkish-cased...
Model agirliklari (outputs/best_model_v3.pt) basariyla yuklendi!


## 1. Girdi ve Tokenizasyon (Subword Splitting)
Modele daha once calisma verisinde (train) gormedigi, Dagilim Disi (OOD) bir cumle veriyoruz.

In [2]:
text = "Bu aksam sinemaya gidecegiz, degil mi?"

inputs = tokenizer(text, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print("Girdi Metin:", text)
print("\nTokenlar:")
print(tokens)

Girdi Metin: Bu aksam sinemaya gidecegiz, degil mi?

Tokenlar:
['[CLS]', 'Bu', 'aksam', 'sine', '##maya', 'gidecegiz', ',', 'degil', 'mi', '?', '[SEP]']


## 2. Model Cikarimi (Inference) ve BIO Etiketleri
Model her bir token icin bir etiket tahmin eder. **CRF (Conditional Random Fields)** katmani sayesinde, `O` etiketinden dogrudan `I-EMPHASIS` etiketine gecis engellenir, once mutlaka `B-EMPHASIS` uretilmesi garantilenir.

In [3]:
# Demo amacli uretilmis model tahmin ciktisi simulasyonu
labels = ['O', 'O', 'O', 'B-EMPHASIS', 'I-EMPHASIS', 'O', 'O', 'O', 'O', 'O', 'O']

html_content = "<table style='width: 40%; text-align: left; font-size: 16px;'><tr><th>Token</th><th>Tahmin (BIO)</th></tr>"
for t, l in zip(tokens, labels):
    if t not in ['[CLS]', '[SEP]']:
        color = '#d32f2f' if 'EMPHASIS' in l else 'black'
        weight = 'bold' if 'EMPHASIS' in l else 'normal'
        html_content += f"<tr><td style='color:{color}; font-weight:{weight}'>{t}</td><td style='color:{color}; font-weight:{weight}'>{l}</td></tr>"
html_content += "</table>"

display(HTML(html_content))

Token,Tahmin (BIO)
Bu,O
aksam,O
sine,B-EMPHASIS
##maya,I-EMPHASIS
gidecegiz,O
",",O
degil,O
mi,O
?,O


## 3. Nihai Vurgu Gosterimi
Subword'ler birlestirilir ve kullaniciya cumle icindeki vurgulu kelime veya obek (span) gosterilir.

In [4]:
def render_emphasis_final(text, emphasis_word):
    highlighted = text.replace(emphasis_word, f"<span style='background-color: #ffeb3b; padding: 2px 8px; border-radius: 4px; font-weight: bold; color: #d32f2f; border: 1px solid #fbc02d; box-shadow: 1px 1px 3px rgba(0,0,0,0.1);'>{emphasis_word}</span>")
    
    html_block = f"""
    <div style='border: 1px solid #ccc; padding: 20px; border-radius: 8px; background-color: #f9f9f9;'>
        <h3 style='margin-top:0;'>Model Ciktisi:</h3>
        <p style='font-size: 22px;'>{highlighted}</p>
    </div>
    """
    display(HTML(html_block))

render_emphasis_final(text, 'sinemaya')